# Project 05: Extreme Value Theory for Tail Risk

This notebook walks through EVT-based tail risk analysis using block maxima (GEV)
and peaks-over-threshold (GPD) approaches.

**Sections:**
1. Load synthetic heavy-tailed data and visualize the distribution
2. Threshold selection: MRL plot, parameter stability, Hill estimator
3. Fit GPD (POT) and GEV (block maxima)
4. Compare EVT VaR vs Normal VaR vs t-VaR at 99%, 99.5%, 99.9%
5. QQ plot and return level curve

**References:**
- de Haan & Ferreira (2006), *Extreme Value Theory: An Introduction*
- McNeil & Frey (2000), Estimation of tail-related risk measures
- Embrechts, Kluppelberg & Mikosch (1997), *Modelling Extremal Events*

In [ ]:
import sys
from pathlib import Path

# Ensure project root and project src/ are importable
_notebook_dir = Path.cwd().resolve()
# Navigate to repo root (handles running from notebooks/ or repo root)
if "05_evt_tail_risk" in str(_notebook_dir):
    _project_dir = _notebook_dir
    while _project_dir.name != "05_evt_tail_risk":
        _project_dir = _project_dir.parent
    _repo_root = _project_dir.parent.parent
else:
    _repo_root = _notebook_dir

_project_src = str(_repo_root / "projects" / "05_evt_tail_risk" / "src")
for p in [str(_repo_root), _project_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Shared library imports
from risk_analyst.models.evt import fit_gev, fit_gpd, evt_var, evt_es, return_level
from risk_analyst.utils.config import load_yaml

# Project-local imports (threshold, model, diagnostics live in project src/)
from threshold import (
    mean_residual_life,
    parameter_stability,
    hill_estimator,
    select_threshold_auto,
)
from model import EVTModel
from diagnostics import (
    plot_qq_gpd,
    plot_return_level,
    plot_evt_vs_normal,
    plot_mean_residual_life,
    plot_threshold_stability,
)

plt.style.use("seaborn-v0_8-whitegrid")
print("All imports successful.")

## 1. Load Synthetic Heavy-Tailed Data and Visualize

We use Pareto(alpha=3) data as a proxy for heavy-tailed financial losses.
The theoretical tail index is xi = 1/alpha = 1/3 ~ 0.333.

This captures the key empirical stylized fact: financial return distributions
have heavier tails than the normal distribution, meaning extreme events occur
more frequently than Gaussian models predict.

In [ ]:
# Generate synthetic heavy-tailed data: Pareto(alpha=3) has xi = 1/3
rng = np.random.default_rng(42)
losses = rng.pareto(3, size=5000)

print(f"Sample size: {len(losses)}")
print(f"Mean:        {np.mean(losses):.4f}")
print(f"Std:         {np.std(losses):.4f}")
print(f"Skewness:    {stats.skew(losses):.4f}")
print(f"Kurtosis:    {stats.kurtosis(losses):.4f}")
print(f"Max:         {np.max(losses):.4f}")
print(f"99th pct:    {np.percentile(losses, 99):.4f}")
print(f"99.9th pct:  {np.percentile(losses, 99.9):.4f}")

In [ ]:
# Visualize distribution: histogram + log-scale tail
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Histogram
axes[0].hist(losses, bins=100, density=True, alpha=0.7, color="#4C72B0",
             edgecolor="white", linewidth=0.3)
axes[0].set_title("Loss Distribution (Pareto)")
axes[0].set_xlabel("Loss")
axes[0].set_ylabel("Density")

# Log-log survival plot (CCDF)
sorted_l = np.sort(losses)[::-1]
ccdf = np.arange(1, len(sorted_l) + 1) / len(sorted_l)
axes[1].loglog(sorted_l, ccdf, ".", markersize=1.5, alpha=0.5)
# Theoretical Pareto CCDF: P(X > x) = (1+x)^{-3}
x_th = np.logspace(-2, np.log10(sorted_l.max()), 200)
axes[1].loglog(x_th, (1 + x_th) ** (-3), "r-", linewidth=1.5, label="Pareto(3) theory")
axes[1].set_title("Log-Log Survival Plot")
axes[1].set_xlabel("Loss (log scale)")
axes[1].set_ylabel("P(X > x) (log scale)")
axes[1].legend()

# QQ against normal -- shows heavy-tail departure
stats.probplot(losses, dist="norm", plot=axes[2])
axes[2].set_title("Normal QQ-Plot")
axes[2].get_lines()[0].set(markersize=2, alpha=0.5)

fig.tight_layout()
plt.show()

## 2. Threshold Selection Diagnostics

Choosing the threshold `u` is the most critical step in POT analysis.
Too low: model includes non-tail data (bias). Too high: too few exceedances (variance).

We use three diagnostic tools:
1. **Mean Residual Life (MRL) plot**: should be approximately linear above the correct threshold
2. **Parameter stability plot**: GPD shape (xi) and modified scale (sigma*) should stabilize
3. **Hill estimator**: non-parametric tail index estimate should plateau

In [ ]:
# 2a. Mean Residual Life plot
thresholds = np.linspace(np.percentile(losses, 50), np.percentile(losses, 98), 50)
thresh_out, mrl = mean_residual_life(losses, thresholds)
fig_mrl, ax_mrl = plot_mean_residual_life(thresh_out, mrl)

# Mark the auto-selected threshold
auto_u = select_threshold_auto(losses, method="percentile", percentile=95.0)
ax_mrl.axvline(auto_u, color="red", linestyle="--", linewidth=1.2, label=f"u = {auto_u:.3f} (95th pct)")
ax_mrl.legend()
plt.show()
print(f"Auto-selected threshold (95th percentile): {auto_u:.4f}")

In [ ]:
# 2b. Parameter stability plot
stability_thresholds = np.linspace(
    np.percentile(losses, 70), np.percentile(losses, 97), 40
)
stab_df = parameter_stability(losses, stability_thresholds)
fig_stab, axes_stab = plot_threshold_stability(stab_df)

# Mark auto threshold
for ax in axes_stab:
    ax.axvline(auto_u, color="red", linestyle="--", linewidth=1.2, label=f"u = {auto_u:.3f}")
axes_stab[0].legend()
plt.show()

In [ ]:
# 2c. Hill estimator
k_range = np.arange(10, 500, 5)
k_vals, hill_est = hill_estimator(losses, k_range)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_vals, hill_est, linewidth=1, color="#4C72B0")
ax.axhline(1.0 / 3.0, color="red", linestyle="--", linewidth=1.2,
           label=r"True $\xi = 1/3$")
ax.set_xlabel("k (number of upper order statistics)")
ax.set_ylabel(r"Hill estimate of $\xi$")
ax.set_title("Hill Estimator Plot")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Report median Hill estimate in stable region
mask = (k_vals >= 100) & (k_vals <= 300)
print(f"Median Hill estimate (k in [100, 300]): {np.median(hill_est[mask]):.4f}")
print(f"True xi = 1/3 = {1/3:.4f}")

## 3. Fit GPD (POT) and GEV (Block Maxima)

Two complementary EVT approaches:

**Peaks-Over-Threshold (POT):** Model exceedances above threshold u with GPD.
Uses more data (all exceedances) but requires threshold selection.

**Block Maxima:** Split data into blocks, take maximum per block, fit GEV.
Avoids threshold choice but uses less data (one observation per block).

In [ ]:
# Load config and create model
config = {
    "pot": {"threshold_percentile": 95.0, "threshold_method": "percentile"},
    "block_maxima": {"block_size": 21},
    "risk": {"confidence_levels": [0.95, 0.99, 0.995, 0.999]},
}
model = EVTModel(config)

# 3a. Peaks-over-threshold: fit GPD
gpd_result = model.fit_pot(losses)
print("=== GPD Fit (Peaks-Over-Threshold) ===")
print(f"  Threshold:    {gpd_result['threshold']:.4f}")
print(f"  Exceedances:  {gpd_result['n_exceed']}")
print(f"  xi (shape):   {gpd_result['xi']:.4f}  (true: 0.3333)")
print(f"  sigma (scale): {gpd_result['sigma']:.4f}")
print()

In [ ]:
# 3b. Block maxima: fit GEV
gev_result = model.fit_block_maxima(losses, block_size=21)
print("=== GEV Fit (Block Maxima) ===")
print(f"  Number of blocks: {gev_result['n_blocks']}")
print(f"  xi (shape):       {gev_result['xi']:.4f}")
print(f"  mu (location):    {gev_result['mu']:.4f}")
print(f"  sigma (scale):    {gev_result['sigma']:.4f}")
print(f"  Neg log-lik:      {gev_result['nll']:.2f}")
print()

# Risk measures at various confidence levels
print("=== EVT Risk Measures ===")
for alpha in [0.95, 0.99, 0.995, 0.999]:
    risk = model.compute_risk(alpha)
    print(f"  alpha={alpha:.3f}: VaR={risk['var_evt']:.4f}, ES={risk['es_evt']:.4f}")

## 4. Compare EVT VaR vs Normal VaR vs t-VaR

The key advantage of EVT: it captures tail behavior that parametric
distributions (Normal, Student-t) underestimate.

At extreme quantiles (99.5%, 99.9%), the gap between EVT and Normal
VaR widens dramatically -- this is precisely where risk managers need
accurate estimates.

In [ ]:
# Compare methods at multiple confidence levels
alphas = [0.95, 0.99, 0.995, 0.999]
comparison_df = model.compare_methods(losses, alphas)
print(comparison_df.to_string(index=False, float_format="{:.4f}".format))
print()

# Relative underestimation by Normal VaR
for _, row in comparison_df.iterrows():
    pct_gap = (row["var_evt"] - row["var_normal"]) / row["var_evt"] * 100
    print(f"  alpha={row['alpha']:.3f}: Normal underestimates EVT VaR by {pct_gap:.1f}%")

In [ ]:
# Grouped bar chart: VaR comparison
fig_cmp, ax_cmp = plot_evt_vs_normal(comparison_df)
plt.show()

## 5. QQ Plot and Return Level Curve

**QQ-Plot:** Points following the 45-degree line indicate a good GPD fit.
Deviations in the upper tail suggest the model may not fully capture extremes.

**Return Level Plot:** Shows the expected maximum loss over a given return period.
For example, the "100-block return level" is the loss we expect to be exceeded
once every 100 blocks (roughly 100 months with monthly blocks).

In [ ]:
# 5a. QQ plot of GPD fit
threshold = gpd_result["threshold"]
exceedances = losses[losses > threshold] - threshold
fig_qq, ax_qq = plot_qq_gpd(exceedances, gpd_result)
plt.show()

In [ ]:
# 5b. GEV return level plot
fig_rl, ax_rl = plot_return_level(gev_result, max_period=200.0)
plt.show()

# Return level table
periods = [2.0, 5.0, 10.0, 25.0, 50.0, 100.0]
rl_df = model.return_levels(periods)
print("Return Level Table:")
print(rl_df.to_string(index=False, float_format="{:.4f}".format))

## Summary

Key findings from this EVT analysis:

1. **GPD fit recovers the true tail index** -- the fitted xi is close to the theoretical 1/3 for Pareto(3) data.
2. **EVT VaR exceeds Normal VaR** at all confidence levels, with the gap widening at extreme quantiles (99.5%, 99.9%). This demonstrates why Gaussian assumptions systematically underestimate tail risk.
3. **Threshold selection diagnostics** (MRL plot, parameter stability, Hill estimator) all confirm the stability of the fitted parameters above the chosen threshold.
4. **Return levels increase monotonically** with return period, as expected from theory.
5. **ES >= VaR** at all confidence levels, consistent with the coherence property of Expected Shortfall.

These EVT tools are essential for regulatory capital calculations (Basel FRTB), insurance ratemaking for catastrophic events, and any risk management application where the far tail matters.